# 8.1 Text preprocessing & normalization

Normalization is the quiet decision about which surface differences in text should count as the same token — made once, before any model sees a character.

Equivalence relations are what a normalizer builds: casefolding and Unicode folding declare that many surface strings name one canonical class. Vocabulary shrinkage is useful only when it does not erase task meaning such as negation, product amounts, or entity case.

Save a copy to Drive to edit.

In [ ]:

import math
import random
import re
import unicodedata
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8)
np.random.seed(8)


def exact_accuracy(predictions, expected):
    correct = 0
    for pred, gold in zip(predictions, expected):
        if pred == gold:
            correct += 1
    return correct / max(1, len(expected))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def show_matrix(ax, matrix, title, xlabels=None, ylabels=None):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels)
    return image


## The concept, built once (D1): a normalization function

We model preprocessing as a function $n:\Sigma^*\to\Sigma^*$, and two strings are equivalent exactly when $n(a)=n(b)$. The lesson toy asks whether `Café`, `CAFÉ`, and `cafe!` collapse from three surface forms to one canonical token.

In [ ]:

raw_text = "Café CAFÉ cafe!"
raw_tokens = raw_text.split()
raw_surface = [token.strip("!") for token in raw_tokens]
raw_vocab = set(raw_surface)

assert len(raw_vocab) == 3

precomposed = "Café"
decomposed = "Cafe\u0301"

assert len(precomposed) == 4
assert len(decomposed) == 5


The reusable method applies Unicode normalization, optional accent stripping, case policy, punctuation policy, and task-specific preservation rules. The important design is not to strip everything; it is to make the equivalence classes match the downstream task.

In [ ]:

def normalize(text, policy):
    protected = policy.get("protected", {})
    keep_negation = policy.get("keep_negation", True)
    keep_digits = policy.get("keep_digits", True)
    keep_case_terms = set(policy.get("keep_case_terms", []))
    words = []
    for raw_token in text.split():
        token = raw_token.strip()
        token = protected.get(token, token)
        if token in keep_case_terms:
            cleaned = token
        else:
            cleaned = unicodedata.normalize("NFKD", token)
            cleaned = "".join(ch for ch in cleaned if not unicodedata.combining(ch))
            cleaned = cleaned.casefold()
        allowed = []
        for ch in cleaned:
            if ch.isalpha():
                allowed.append(ch)
            elif ch.isdigit() and keep_digits:
                allowed.append(ch)
            elif ch in {"$", "%"} and keep_digits:
                allowed.append(ch)
        cleaned = "".join(allowed)
        if cleaned in {"not", "no", "never"} and not keep_negation:
            continue
        if cleaned:
            words.append(cleaned)
    return " ".join(words)

lesson_policy = {"keep_digits": True, "keep_negation": True}
lesson_norm = normalize(raw_text, lesson_policy)
lesson_vocab = set(lesson_norm.split())

assert lesson_norm == "cafe cafe cafe"
assert len(lesson_vocab) == 1
assert unicodedata.normalize("NFC", decomposed) == precomposed


## Dataset ladder: inline D1–D5 text/token data with rising complexity

In [ ]:

normalization_ladder = [
    {
        "name": "D1 lesson cafe toy",
        "items": [("Café CAFÉ cafe!", "cafe cafe cafe")],
    },
    {
        "name": "D2 five clean query variants",
        "items": [
            ("CAFÉ deals", "cafe deals"),
            ("Resume tips", "resume tips"),
            ("naïve Bayes", "naive bayes"),
            ("São Paulo", "sao paulo"),
            ("co-op guide", "coop guide"),
        ],
    },
    {
        "name": "D3 structure with negation case and digits",
        "items": [
            ("US launch not delayed", "US launch not delayed"),
            ("Save $5000 now", "save $5000 now"),
            ("Apple is not apple", "Apple is not apple"),
            ("never drop SKU5000", "never drop sku5000"),
            ("No refunds for Plan-2", "no refunds for plan2"),
        ],
    },
    {
        "name": "D4 inline support-ticket titles",
        "items": [
            ("CAFÉ receipt failed for US account", "cafe receipt failed for US account"),
            ("User says not billed $5000", "user says not billed $5000"),
            ("Résumé upload broken on iOS", "resume upload broken on ios"),
            ("Apple campaign ID 5000 missing", "Apple campaign id 5000 missing"),
            ("São Paulo invoice no longer visible", "sao paulo invoice no longer visible"),
        ],
    },
    {
        "name": "D5 longer mixed product strings",
        "items": [
            ("US seller says Café campaign is not eligible for $5000 credit", "US seller says cafe campaign is not eligible for $5000 credit"),
            ("Apple reseller in São Paulo never received SKU5000 résumé bundle", "Apple reseller in sao paulo never received sku5000 resume bundle"),
            ("No refund when Plan-2 invoice shows CAFÉ add-on", "no refund when plan2 invoice shows cafe addon"),
            ("Support note: US means country, us means pronoun, not same", "support note US means country us means pronoun not same"),
            ("Price parser dropped $5000 from café-promo email", "price parser dropped $5000 from cafepromo email"),
        ],
    },
]

for rung in normalization_ladder:
    print(rung["name"], "items", len(rung["items"]))
    print("sample", rung["items"][0])


In [ ]:

field_policy = {
    "keep_digits": True,
    "keep_negation": True,
    "keep_case_terms": {"US", "Apple"},
}

results = []
for index, rung in enumerate(normalization_ladder, start=1):
    predictions = [normalize(text, field_policy) for text, gold in rung["items"]]
    expected = [gold for text, gold in rung["items"]]
    acc = exact_accuracy(predictions, expected)
    results.append({"rung": index, "name": rung["name"], "accuracy": acc})

for row in results:
    print(row["rung"], row["name"], f"accuracy={row['accuracy']:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, rung in enumerate(normalization_ladder):
    raw, gold = rung["items"][0]
    normalized = normalize(raw, field_policy)
    raw_tokens = raw.split()
    norm_tokens = normalized.split()
    matrix = np.zeros((len(norm_tokens), len(raw_tokens)))
    for i, norm_token in enumerate(norm_tokens):
        for j, raw_token in enumerate(raw_tokens):
            test_token = normalize(raw_token, field_policy)
            matrix[i, j] = float(test_token == norm_token)
    show_matrix(axes[0, idx], matrix, f"D{idx + 1}", raw_tokens, norm_tokens)

x = [row["rung"] for row in results]
y = [row["accuracy"] for row in results]
axes[1, 0].plot(x, y, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("Accuracy vs complexity")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: over-normalization erases the signal

In [ ]:

bad_policy = {
    "keep_digits": False,
    "keep_negation": False,
    "keep_case_terms": set(),
}

hard_text, hard_gold = normalization_ladder[-1][0]
bad = normalize(hard_text, bad_policy)
fixed = normalize(hard_text, field_policy)

print("raw:", hard_text)
print("bad:", bad)
print("fixed:", fixed)
print("gold:", hard_gold)
print("bad correct", bad == hard_gold)
print("fixed correct", fixed == hard_gold)


## Evaluate it + Practice

- Metric: normalization accuracy against expected canonical forms on D1–D5
- No-skill baseline: lowercase and drop all nonletters, which loses US, not, and $5000
- Cheap sanity check: the D1 toy must shrink 3 surface variants to 1 token
- Ablation: turn off protected case or digit rules and watch D5 accuracy drop
- Failure signals: vocabulary unexpectedly grows, negation vanishes, or visually identical Unicode strings split counts

### Practice

**Practice:** Add one D5 example with a code token such as SKU-77 and decide what the canonical form should preserve.

**Practice:** Change the field policy so product names keep case and compare the D4 accuracy.